The Ellipticity-Shear Correlation
=================================

Credit: Bekah Polen (Duke)

A demonstration of the relationship between ellipticity ($\epsilon$) and shear ($\gamma$). Show that observed galaxy ellipticities are unbiased estimators of shear in the weak lensing limit: where $\lvert \gamma \rvert \ll 1$ and when ensemble-averaged. Or, that: $$\langle \epsilon_{obs} \rangle = \gamma$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

Generate a population of intrinsic ellipticities:
-------------------------------------------------

In [ ]:
# Number of galaxies
N = 10000
# Random ellipticity magnitudes (typical galaxy ellipticity distribution)
e = np.random.uniform(0, 0.5, N)
# Random orientation angles
phi = np.random.uniform(0, 2*np.pi, N)
# Complex ellipticities
eps_int = e * np.exp(2j * phi)
print("Mean intrinsic ellipticity:", np.mean(eps_int))

# Plot
plt.figure(figsize=(5,5))
plt.scatter(eps_int.real, eps_int.imag, s=2, alpha=0.2)
plt.axhline(0, color='k', lw=1)
plt.axvline(0, color='k', lw=1)
plt.title("Intrinsic Ellipticity Distribution")
plt.xlabel("Re(ε)")
plt.ylabel("Im(ε)")
plt.show()

Apply a constant shear:
-----------------------

Apply a shear signal to the intrinsic ellipticities.

In [ ]:
gamma = 0.01 + 0.01j

# Linearized expansion valid in weak lensing regime
eps_obs = eps_int + gamma

print("Mean observed ellipticity:", np.mean(eps_obs))
print("Input shear:", gamma)
print("Difference:", np.mean(eps_obs) - gamma)

Now plot observed ellipticity and notice almost no visual change:
-----------------------------------------------------------------

In [ ]:
# Plot
plt.figure(figsize=(5,5))
plt.scatter(eps_obs.real, eps_obs.imag, s=2, alpha=0.2)
plt.axhline(0, color='k', lw=1)
plt.axvline(0, color='k', lw=1)
plt.title("Observed Ellipticity Distribution")
plt.xlabel("Re(ε)")
plt.ylabel("Im(ε)")
plt.show()

Note that we are working in complex space.
------------------------------------------

In [ ]:
# Intrinsic ellipticities
N = 5000
e = np.random.uniform(0, 0.5, N)
phi = np.random.uniform(0, 2*np.pi, N)
eps_int = e * np.exp(2j*phi)

# Shear
gamma = 0.01 + 0.01j

# Observed ellipticities
eps_obs = eps_int + gamma

# Plot
plt.figure(figsize=(6,6))
plt.scatter(eps_int.real, eps_int.imag, s=5, alpha=0.3, label="Intrinsic ε (mean=0)")
plt.scatter(eps_obs.real, eps_obs.imag, s=5, alpha=0.3, label="Observed ε = ε_int + γ")
plt.plot(gamma.real, gamma.imag, 'rx', markersize=12, mew=3, label="Shear γ")
plt.axhline(0, color='k', lw=1)
plt.axvline(0, color='k', lw=1)
plt.xlabel("Re(ε)")
plt.ylabel("Im(ε)")
plt.title("Intrinsic vs Observed Ellipticities in the Weak-Shear Limit")
# Set axis limits so gamma is visible
lim = 0.6  # wide enough to see the whole cloud
plt.xlim(-lim, lim)
plt.ylim(-lim, lim)

plt.legend()
plt.grid(True)
plt.gca().set_aspect('equal')
plt.show()

Example of an Ensemble Average:
-------------------------------

When averaging over many galaxy shapes in the weak lensing regime, the ellipticity is equivalent to the shear. Note that the input `gamma1` and `gamma2` values reflect the resulting `|<e>|` value. We are assuming no intrinsic alignment here, such that the random intrinsic shapes are truly randomly oriented. 

In [ ]:
N = 60
gamma1, gamma2 = 0.1, 0.0
a_fixed = 0.15               # radius of original circles
A_fixed = np.pi * a_fixed**2 # FIX the area for each galaxy
np.random.seed(12)

# Generate intrinsic ellipticities
q_intrinsic = np.random.uniform(0.6, 1.0, N)
phi = np.random.uniform(0, 2*np.pi, N)

# Intrinsic complex ellipticity
e_int = (1 - q_intrinsic) / (1 + q_intrinsic) * np.exp(2j * phi)

# Apply shear (complex ellipticity transformation)
gamma = gamma1 + 1j * gamma2
e_obs = (e_int + gamma) / (1 + gamma * np.conjugate(e_int))

def e_to_q_and_angle(e):
    """Return axis ratio q and orientation angle (degrees) from complex ellipticity."""
    mag = np.abs(e)
    # clip to avoid q<=0
    if mag >= 1:
        mag = 0.999
    q = (1 - mag) / (1 + mag)
    angle = 0.5 * np.angle(e) * 180/np.pi
    return q, angle

def circularity_stats(e_arr):
    e_mean = np.mean(e_arr)
    e_amp = np.abs(e_mean)
    q_eff = (1 - e_amp) / (1 + e_amp)
    C = 1 - e_amp
    return e_amp, q_eff, C, e_mean

stats_int = circularity_stats(e_int)
stats_obs = circularity_stats(e_obs)

# Plot
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
titles = ["Random Intrinsic Shapes", "After Shear Applied"]

for i, (e_arr, axis, stats) in enumerate(zip([e_int, e_obs], ax, [stats_int, stats_obs])):

    areas = []
    # Draw many ellipses (all centered at 0,0) but with transparency
    for e in e_arr:
        q, ang = e_to_q_and_angle(e)
        # compute semimajor a and semiminor b so that area fixed
        a = np.sqrt(A_fixed / (np.pi * q))
        b = a * q
        area = np.pi * a * b
        areas.append(area)
        ell = Ellipse((0, 0), 2*a, 2*b, angle=ang,
                      edgecolor='blue', facecolor='none', lw=0.6, alpha=0.25)
        axis.add_patch(ell)

    # Mean ellipse using mean complex ellipticity but same fixed area
    e_mean = stats[3]
    q_mean, angle_mean = e_to_q_and_angle(e_mean)
    mean_area = np.mean(areas)  # should be ~ A_fixed, but use actual mean to be robust
    a_mean = np.sqrt(mean_area / (np.pi * q_mean))
    b_mean = a_mean * q_mean

    mean_ell = Ellipse((0, 0), 2*a_mean, 2*b_mean, angle=angle_mean,
                       edgecolor='blue', facecolor='none', lw=2)
    axis.add_patch(mean_ell)

    axis.set_xlim(-0.4, 0.4)
    axis.set_ylim(-0.4, 0.4)
    axis.set_aspect("equal")
    axis.set_xticks([])
    axis.set_yticks([])
    axis.set_title(titles[i], fontsize=14)

# Print ellipticity information
labels = [
    fr"$| \langle e \rangle | = {stats_int[0]:.3f}$" + "\n"
    + fr"$q = {stats_int[1]:.3f}$" + "\n"
    + fr"$C = {stats_int[2]:.3f}$",

    fr"$| \langle e \rangle | = {stats_obs[0]:.3f}$" + "\n"
    + fr"$q = {stats_obs[1]:.3f}$" + "\n"
    + fr"$C = {stats_obs[2]:.3f}$"
]

for i, axis in enumerate(ax):
    axis.text(0.5, -0.15, labels[i], ha='center', va='top',
              fontsize=12, transform=axis.transAxes)

plt.tight_layout()
plt.show()

As galaxy sample size increases, the shear estimator converges to $\gamma$:
---------------------------------------------------------------------------
The shear estimator = mean ellipticities, or in this case, 0.

In [ ]:
Ns = [10, 100, 1000, 5000, 10000, 50000, 100000, 1000000,
      10000000, 100000000]
gamma = 0.02 + 0.01j
errors = []

for N in Ns:
    # regenerate intrinsic shapes for each N
    e = np.random.uniform(0, 0.5, N)
    phi = np.random.uniform(0, 2*np.pi, N)
    eps_int = e * np.exp(2j * phi)

    eps_obs = eps_int + gamma
    gamma_hat = np.mean(eps_obs)

    errors.append(np.abs(gamma_hat - gamma))

plt.figure(figsize=(6,4))
plt.loglog(Ns, errors, marker='o')
plt.xlabel("Number of galaxies")
plt.ylabel("|γ̂ − γ|")
plt.title("Shear Estimation Error vs Sample Size")
plt.grid(True)
plt.show()

Understanding the weak lensing approximation:
---------------------------------------------

$\epsilon_{obs} = \frac{\epsilon_{int}+\gamma}{1+\gamma^{*}\epsilon_{int}}$
------------------------------------------------------------------------

A demonstration of how the weak lensing approximation breaks down in stronger shear regimes.

In [ ]:
# Regenerate a large sample of intrinsic ellipticities
N = 50000
e = np.random.uniform(0, 0.5, N)
phi = np.random.uniform(0, 2*np.pi, N)
eps_int = e * np.exp(2j * phi)

# Two gamma ranges
gamma_ranges = [
    np.linspace(0.0, 0.1, 20),  # weak regime
    np.linspace(0.0, 1, 20)   # strong regime
]

titles = [
    "Weak Shear Regime (γ ≤ 0.1)",
    "Strong Shear Regime (γ ≤ 1)"
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, gammas, title in zip(axes, gamma_ranges, titles):
    means_weak = []
    means_exact = []
    for g in gammas:
        gamma = g + 0j  # real shear
        # Weak approximation
        eps_obs_weak = eps_int + gamma
        # Exact shear mapping
        eps_obs_exact = (eps_int + gamma) / (1 + gamma * np.conj(eps_int))
        means_weak.append(np.mean(eps_obs_weak).real)
        means_exact.append(np.mean(eps_obs_exact).real)

    # Plot
    ax.plot(gammas, means_exact, 'o-', label="Exact")
    ax.plot(gammas, means_weak, 's--', label="Weak Approx.")
    ax.plot(gammas, gammas, 'k:', label="Input γ")
    ax.set_xlabel("Input shear γ")
    ax.set_title(title)
    ax.grid(True)

# Shared y-label
axes[0].set_ylabel("Measured <ε>")
# Place legend only once, below the plots
fig.legend(loc="lower center", ncol=3)
fig.tight_layout(rect=[0, 0.12, 1, 1])
plt.show()